## First instal exiftools

## Necessary Imports

In [1]:
import json
import subprocess
from pathlib import Path
from datetime import datetime
from datetime import datetime, date

## Set your paths here

In [2]:
exiftool_path="path_to_your_exif.exe"
example_file="path to one example file"
JSON_extension=".supplemental-metadata.json"

In [3]:
def show_json_metadata(file_path):
    json_path = Path(str(file_path) + JSON_extension)

    if not json_path.exists():
        raise FileNotFoundError(json_path)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    import pprint
    pprint.pprint(data, sort_dicts=False)

def merge_metadata(media_file, debug=False):
    media_file = Path(media_file)
    json_file = Path(str(media_file) + JSON_extension)

    if not json_file.exists():
        raise FileNotFoundError(f"JSON not found: {json_file}")

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    cmd = [exiftool_path, "-overwrite_original"]

    # Photo/video timestamp
    ts = data.get("photoTakenTime", {}).get("timestamp")
    if ts:
        dt = datetime.utcfromtimestamp(int(ts))
        dt_str = dt.strftime("%Y:%m:%d %H:%M:%S")
        cmd += [
            f"-DateTimeOriginal={dt_str}",
            f"-CreateDate={dt_str}",
            f"-ModifyDate={dt_str}",
        ]

    # GPS
    geo = data.get("geoData", {})
    lat = geo.get("latitude")
    lon = geo.get("longitude")

    if lat not in (None, 0) and lon not in (None, 0):
        cmd += [
            f"-GPSLatitude={lat}",
            f"-GPSLongitude={lon}",
        ]

    # Description
    if data.get("description"):
        cmd.append(f"-Description={data['description']}")

    # Title
    if data.get("title"):
        cmd.append(f"-Title={data['title']}")

    cmd.append(str(media_file))

    if debug:
        print("Running:")
        print(" ".join(cmd))

    result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
    )

    if debug:
        print("STDOUT:")
        print(result.stdout)
        
        print("STDERR:")
        print(result.stderr)
        
        print("Return code:", result.returncode)
    
        print(f"✓ Updated {media_file}")

def process_takeout_folder(big_folder, func, was_interrupted=False):
    big_folder = Path(big_folder)

    # common media extensions (images + videos)
    media_ext = {
        ".jpg", ".jpeg", ".png", ".heic",
        ".mp4", ".mov", ".avi", ".mkv", ".3gp", ".webm"
    }

    success = 0
    skipped = 0
    failed = 0

    if not was_interrupted:
        for file in big_folder.rglob("*"):
            if not file.is_file():
                continue
    
            if file.suffix.lower() not in media_ext:
                continue
    
            json_file = Path(str(file) + JSON_extension)
    
            if not json_file.exists():
                skipped += 1
                continue
    
            try:
                func(file)
                success += 1
    
            except Exception as e:
                print(f"❌ Failed: {file}\n   → {e}\n")
                failed += 1
    else:
        today = date.today()
        for folder in big_folder.rglob("*"):

            if not folder.is_dir():
                continue
        
            folder_date = datetime.fromtimestamp(
                folder.stat().st_mtime
            ).date()
        
            # Skip folders touched today
            if folder_date == today:
                continue
        
            print(f"Processing folder: {folder}")
        
            for file in folder.iterdir():
        
                if not file.is_file():
                    continue
        
                if file.suffix.lower() not in media_ext:
                    continue
        
                if not Path(str(file) + JSON_extension).exists():
                    continue
        
                try:
                    func(file)
                    print(f"  ✓ {file.name}")
        
                except Exception as e:
                    print(f"  ✗ {file.name}: {e}")

    print("\n===== SUMMARY =====")
    print("Processed successfully:", success)
    print("Skipped (no JSON):     ", skipped)
    print("Failed:                ", failed)

In [4]:
show_json_metadata(example_file)

FileNotFoundError: path to one example file.supplemental-metadata.json

In [ ]:
merge_metadata(example_file)

In [ ]:
process_takeout_folder(r"D:\Albums_output", merge_metadata, was_interrupted=True)

## IMPORTANT CHECK VIDEOS Seem they are not treated correctly

In [5]:
from pathlib import Path
from datetime import datetime, date

MEDIA_EXTS = {
    ".jpg", ".jpeg", ".png", ".heic",
    ".mp4", ".mov", ".avi", ".mkv", ".webm", ".3gp"
}

root = Path(r"D:\Albums_output")
today = date.today()

for folder in root.rglob("*"):

    if not folder.is_dir():
        continue

    folder_date = datetime.fromtimestamp(
        folder.stat().st_mtime
    ).date()

    # Skip folders touched today
    if folder_date == today:
        continue

    print(f"Processing folder: {folder}")

    for file in folder.iterdir():

        if not file.is_file():
            continue

        if file.suffix.lower() not in MEDIA_EXTS:
            continue

        if not Path(str(file) + ".json").exists():
            continue

        try:
            func(file)
            print(f"  ✓ {file.name}")

        except Exception as e:
            print(f"  ✗ {file.name}: {e}")